# Working with Text Data

In [1]:
import pandas as pd

## This Module's Dataset
- This module's dataset (`chicago.csv`) is a collection of public sector employees in the city of Chicago.
- Each row includes the employee's name, position, department, and salary.

In [2]:
chicago = pd.read_csv("chicago.csv").dropna(how="all")
chicago.head()

,Name,Position Title,Department,Employee Annual Salary
0,"AARON, ELVIA J",WATER RATE TAKER,WATER MGMNT,$90744.00
1,"AARON, JEFFERY M",POLICE OFFICER,POLICE,$84450.00
2,"AARON, KARINA",POLICE OFFICER,POLICE,$84450.00
3,"AARON, KIMBERLEI R",CHIEF CONTRACT EXPEDITER,GENERAL SERVICES,$89880.00
4,"ABAD JR, VICENTE M",CIVIL ENGINEER IV,WATER MGMNT,$106836.00


In [3]:
chicago.describe()

,Name,Position Title,Department,Employee Annual Salary
count,32062,32062,32062,32062
unique,31776,1093,35,1156
top,"HERNANDEZ, JUAN C",POLICE OFFICER,POLICE,$87384.00
freq,4,9184,12618,2394


## Common String Methods
- A `Series` has a special `str` attribute that exposes an object with string methods.
- Access the `str` attribute, then invoke the string method on the nested object.
- Most method names will match their Python method equivalents (`upper`, `lower`, `title`, etc).

In [4]:
chicago["Position Title"].str.len()

0        16
1        14
2        14
3        24
4        17
         ..
32057    30
32058    14
32059    14
32060    14
32061    23
Name: Position Title, Length: 32062, dtype: int64

- The `str` attribute exposes an object that supports square bracket syntax for indexing.
- You can provide values, slices, or negative numbers.

In [5]:
chicago["Position Title"].str[3:10]

0        ER RATE
1        ICE OFF
2        ICE OFF
3        EF CONT
4        IL ENGI
          ...   
32057     OF MAC
32058    ICE OFF
32059    ICE OFF
32060    ICE OFF
32061    EF DATA
Name: Position Title, Length: 32062, dtype: str

## Filtering with String Methods
- The `str.contains` method checks whether a substring exists anywhere in the string.
- The `str.startswith` method checks whether a substring exists at the start of the string.
- The `str.endswith` method checks whether a substring exists at the end of the string.

In [10]:
chicago["Position Title"].str.contains("POLICE")

0        False
1         True
2         True
3        False
4        False
         ...  
32057    False
32058     True
32059     True
32060     True
32061    False
Name: Position Title, Length: 32062, dtype: bool

In [9]:
chicago["Position Title"].str.startswith("WATER") # type sensitive!

0         True
1        False
2        False
3        False
4        False
         ...  
32057    False
32058    False
32059    False
32060    False
32061    False
Name: Position Title, Length: 32062, dtype: bool

In [12]:
# Can also use this as a search query, but .str chaining is weird
is_police = chicago["Position Title"].str.lower().str.startswith("police")
chicago[is_police].head()

,Name,Position Title,Department,Employee Annual Salary
1,"AARON, JEFFERY M",POLICE OFFICER,POLICE,$84450.00
2,"AARON, KARINA",POLICE OFFICER,POLICE,$84450.00
11,"ABBATE, TERRY M",POLICE OFFICER,POLICE,$90618.00
15,"ABDALLAH, ZAID",POLICE OFFICER,POLICE,$74028.00
16,"ABDELHADI, ABDALMAHD",POLICE OFFICER,POLICE,$81588.00


## String Methods on Index and Columns
- Use the `index` and `columns` attributes to access the `DataFrame` index/column labels.
- These objects support string methods via their own `str` attribute.

In [13]:
chicago_str_index = (
    pd.read_csv("chicago.csv", index_col="Name")
    .dropna(how="all")
    .astype({"Department": "category"})
)
chicago_str_index.head()

,Position Title,Department,Employee Annual Salary
Name,,,
"AARON, ELVIA J",WATER RATE TAKER,WATER MGMNT,$90744.00
"AARON, JEFFERY M",POLICE OFFICER,POLICE,$84450.00
"AARON, KARINA",POLICE OFFICER,POLICE,$84450.00
"AARON, KIMBERLEI R",CHIEF CONTRACT EXPEDITER,GENERAL SERVICES,$89880.00
"ABAD JR, VICENTE M",CIVIL ENGINEER IV,WATER MGMNT,$106836.00


In [14]:
chicago_str_index.index = chicago_str_index.index.str.title()

In [15]:
chicago_str_index.head()

,Position Title,Department,Employee Annual Salary
Name,,,
"Aaron, Elvia J",WATER RATE TAKER,WATER MGMNT,$90744.00
"Aaron, Jeffery M",POLICE OFFICER,POLICE,$84450.00
"Aaron, Karina",POLICE OFFICER,POLICE,$84450.00
"Aaron, Kimberlei R",CHIEF CONTRACT EXPEDITER,GENERAL SERVICES,$89880.00
"Abad Jr, Vicente M",CIVIL ENGINEER IV,WATER MGMNT,$106836.00


## The str.split Method
- The `str.split` method splits a string by the occurrence of a delimiter. Pandas returns a `Series` of lists.
- Use the `str.get` method to access a nested list element by its index position.

In [17]:
new_serieses = chicago["Position Title"].str.split(" ")
new_serieses.head()

0            [WATER, RATE, TAKER]
1               [POLICE, OFFICER]
2               [POLICE, OFFICER]
3    [CHIEF, CONTRACT, EXPEDITER]
4           [CIVIL, ENGINEER, IV]
Name: Position Title, dtype: object

In [18]:
new_serieses[0]

['WATER', 'RATE', 'TAKER']

In [19]:
new_serieses.value_counts()

Position Title
[POLICE, OFFICER]                               9184
[FIREFIGHTER-EMT]                               1208
[SERGEANT]                                      1185
[POOL, MOTOR, TRUCK, DRIVER]                     918
[POLICE, OFFICER, (ASSIGNED, AS, DETECTIVE)]     896
                                                ... 
[DECK, HAND]                                       1
[CASE, ANALYST, -, LAW]                            1
[OPERATIONS, ANALYST]                              1
[PREPRESS, TECHNICIAN]                             1
[MECHANICAL, ENGINEER, IV]                         1
Name: count, Length: 1093, dtype: int64

- Let's find the most common first name among the employees.

In [20]:
chicago.Name

0            AARON,  ELVIA J
1          AARON,  JEFFERY M
2             AARON,  KARINA
3        AARON,  KIMBERLEI R
4        ABAD JR,  VICENTE M
                ...         
32057    ZYGADLO,  MICHAEL J
32058     ZYGOWICZ,  PETER J
32059      ZYMANTAS,  MARK E
32060    ZYRKOWSKI,  CARLO E
32061    ZYSKOWSKI,  DARIUSZ
Name: Name, Length: 32062, dtype: str

In [25]:
# ok, so first name comes after the ", "
first_names = chicago.Name.str.split(", ").str.get(1).str.strip().str.split(" ").str.get(0)
first_names

0            ELVIA
1          JEFFERY
2           KARINA
3        KIMBERLEI
4          VICENTE
           ...    
32057      MICHAEL
32058        PETER
32059         MARK
32060        CARLO
32061      DARIUSZ
Name: Name, Length: 32062, dtype: object

In [26]:
first_names.value_counts()

Name
MICHAEL    1153
JOHN        899
JAMES       676
ROBERT      622
JOSEPH      537
           ... 
SANDINO       1
HAMZEH        1
JANAAN        1
ALEN          1
MAC           1
Name: count, Length: 5091, dtype: int64

## The expand and n Parameters of the str.split Method
- The `expand` parameter returns a `DataFrame` instead of a `Series` of lists.
- The `n` parameter limits the number of splits.

In [27]:
new_frame = chicago["Name"].str.title().str.split(", ", expand=True)
new_frame

,0,1
0,Aaron,Elvia J
1,Aaron,Jeffery M
2,Aaron,Karina
3,Aaron,Kimberlei R
4,Abad Jr,Vicente M
...,...,...
32057,Zygadlo,Michael J
32058,Zygowicz,Peter J
32059,Zymantas,Mark E
32060,Zyrkowski,Carlo E


In [29]:
# now we have a data frame, but we need to map it to new columns in the original
# This feels like destructuring
chicago[["Last Name", "First Name"]] = new_frame
chicago.head()

,Name,Position Title,Department,Employee Annual Salary,Last Name,First Name
0,"AARON, ELVIA J",WATER RATE TAKER,WATER MGMNT,$90744.00,Aaron,Elvia J
1,"AARON, JEFFERY M",POLICE OFFICER,POLICE,$84450.00,Aaron,Jeffery M
2,"AARON, KARINA",POLICE OFFICER,POLICE,$84450.00,Aaron,Karina
3,"AARON, KIMBERLEI R",CHIEF CONTRACT EXPEDITER,GENERAL SERVICES,$89880.00,Aaron,Kimberlei R
4,"ABAD JR, VICENTE M",CIVIL ENGINEER IV,WATER MGMNT,$106836.00,Abad Jr,Vicente M


## The explode Method
- The `explode` method transforms a `Series` of lists into a column with a separate row for each list element.
- Pandas preserves the original index. Use `reset_index` with `drop=True` to generate a new sequential index.

In [30]:
skills = pd.read_csv("employee_skills.csv")
skills.head()

,Employee,Department,Skills
0,Alice,Engineering,Python|SQL|Docker
1,Bob,Engineering,Java|Python|R
2,Carol,Data Science,SQL
3,David,Data Science,Python|SQL|Tableau
4,Emma,Engineering,Python|JavaScript


In [31]:
# then convert that Skills series into a 2D one
skills["Skills"] = skills["Skills"].str.split("|")
skills.head()

,Employee,Department,Skills
0,Alice,Engineering,"[Python, SQL, Docker]"
1,Bob,Engineering,"[Java, Python, R]"
2,Carol,Data Science,[SQL]
3,David,Data Science,"[Python, SQL, Tableau]"
4,Emma,Engineering,"[Python, JavaScript]"


In [ ]:
skills.explode("Skills") # whoa, this creates duplicative indexes...

,Employee,Department,Skills
0,Alice,Engineering,Python
0,Alice,Engineering,SQL
0,Alice,Engineering,Docker
1,Bob,Engineering,Java
1,Bob,Engineering,Python
1,Bob,Engineering,R
2,Carol,Data Science,SQL
3,David,Data Science,Python
3,David,Data Science,SQL
3,David,Data Science,Tableau


In [34]:
# this makes sure new indexes are assigned
skills.explode("Skills").reset_index(drop=True)

,Employee,Department,Skills
0,Alice,Engineering,Python
1,Alice,Engineering,SQL
2,Alice,Engineering,Docker
3,Bob,Engineering,Java
4,Bob,Engineering,Python
5,Bob,Engineering,R
6,Carol,Data Science,SQL
7,David,Data Science,Python
8,David,Data Science,SQL
9,David,Data Science,Tableau
